In [ ]:
import numpy as np
import torch.optim as optim
import os
from sklearn.metrics import f1_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

In [ ]:
class ECGDataset(Dataset):
    def __init__(self, data_dir, prefix_dwt='X_dwt_', prefix_stft='X_stft_', prefix_y='y_'):
        self.data_dir = data_dir
        
        # Собираем список всех файлов батчей
        self.dwt_files = sorted([f for f in os.listdir(data_dir) if f.startswith(prefix_dwt)])
        self.stft_files = sorted([f for f in os.listdir(data_dir) if f.startswith(prefix_stft)])
        self.y_files = sorted([f for f in os.listdir(data_dir) if f.startswith(prefix_y)])
        
        self.cumulative_sizes = []
        current_total = 0
        
        # Индексируем датасет (считаем, сколько окон в каждом файле)
        for f in self.y_files:
            labels = np.load(os.path.join(data_dir, f))
            current_total += len(labels)
            self.cumulative_sizes.append(current_total)
        
        self.total_size = current_total
        # Кэш для последнего загруженного файла (чтобы не читать диск на каждом шаге)
        self.current_batch_idx = -1
        self.batch_data = (None, None, None)

    def __len__(self):
        return self.total_size

    def _load_batch_for_index(self, global_idx):
        # Определяем, в каком файле лежит этот индекс
        batch_idx = next(i for i, size in enumerate(self.cumulative_sizes) if size > global_idx)
        
        # Если файл уже в памяти, ничего не делаем
        if batch_idx == self.current_batch_idx:
            return batch_idx
            
        # Загружаем новые данные
        dwt = np.load(os.path.join(self.data_dir, self.dwt_files[batch_idx]))
        stft = np.load(os.path.join(self.data_dir, self.stft_files[batch_idx]))
        y = np.load(os.path.join(self.data_dir, self.y_files[batch_idx]))
        
        self.batch_data = (dwt, stft, y)
        self.current_batch_idx = batch_idx
        return batch_idx

    def __getitem__(self, idx):
        batch_idx = self._load_batch_for_index(idx)
        
        # Вычисляем локальный индекс внутри батча
        start_idx = self.cumulative_sizes[batch_idx - 1] if batch_idx > 0 else 0
        local_idx = idx - start_idx
        
        d_sample, s_sample, y_sample = self.batch_data
        
        # Извлекаем данные
        x_dwt = d_sample[local_idx]
        x_stft = s_sample[local_idx]
        target = y_sample[local_idx]
        
        x_dwt = torch.from_numpy(x_dwt).permute(2, 0, 1).float()
        x_stft = torch.from_numpy(x_stft).permute(2, 0, 1).float()
        target = torch.from_numpy(target).float()
        
        return x_dwt, x_stft, target

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        residual = self.shortcut(x)
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += residual
        return F.relu(out)

class SEAttention(nn.Module):
    # Простой механизм внимания для взвешивания каналов
    def __init__(self, channels, reduction=4):
        super(SEAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

class ECGMultiModalNet(nn.Module):
    def __init__(self, num_classes):
        super(ECGMultiModalNet, self).__init__()
        
        # Ветка DWT
        self.dwt_branch = nn.Sequential(
            nn.Conv2d(8, 32, kernel_size=(3, 7), padding=(1, 3)),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            ResidualBlock(32, 32),
            SEAttention(32),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten()
        )
        
        # Ветка STFT
        self.stft_branch = nn.Sequential(
            nn.Conv2d(8, 32, kernel_size=(3, 3), padding=(1, 1)),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            ResidualBlock(32, 32),
            SEAttention(32),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten()
        )
        
        # Общий классификатор
        self.classifier = nn.Sequential(
            nn.Linear(32 + 32, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x_dwt, x_stft):
        feat_dwt = self.dwt_branch(x_dwt)
        feat_stft = self.stft_branch(x_stft)
        
        combined = torch.cat((feat_dwt, feat_stft), dim=1)
        return self.classifier(combined)

In [ ]:
def train_model(train_loader, val_loader, device, epochs=10, threshold=0.5):
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        
        all_preds = []
        all_labels = []
        
        for x_dwt, x_stft, labels in train_loader:
            x_dwt, x_stft, labels = x_dwt.to(device), x_stft.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(x_dwt, x_stft)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            
            # Переводим логиты в вероятности и применяем порог
            preds = (torch.sigmoid(outputs) > threshold).int()
            
            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
            
        # Склеиваем всё в один массив для расчета метрик
        all_preds = np.vstack(all_preds)
        all_labels = np.vstack(all_labels)
        
        # Считаем F1 (macro — среднее по каждому из 72 классов)
        train_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
        
        # Валидация
        val_loss, val_f1 = validate(model, val_loader, criterion, device, threshold)
        
        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]['lr']
        
        checkpoint = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'val_f1': val_f1,
            'val_loss': val_loss
        }

        # 1. Сохраняем последний чекпоинт (перезаписывается каждую эпоху)
        torch.save(checkpoint, 'last_checkpoint.pth')

        # 2. Если модель стала лучше — сохраняем её как "лучшую"
        # if val_f1 > best_val_f1:
        #     best_val_f1 = val_f1
        #     torch.save(checkpoint, 'best_model.pth')
        #     print(f"!!! New best model saved (F1: {val_f1:.4f}) !!!")
        
        print(f"Epoch {epoch+1} | LR: {current_lr:.6f}")
        print(f"Train Loss: {running_loss/len(train_loader):.4f} | Train F1: {train_f1:.4f}")
        print(f"Val Loss: {val_loss:.4f} | Val F1: {val_f1:.4f}")
        print("-" * 30)

def validate(model, loader, criterion, device, threshold=0.5):
    model.eval()
    val_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for x_dwt, x_stft, labels in loader:
            x_dwt, x_stft, labels = x_dwt.to(device), x_stft.to(device), labels.to(device)
            
            outputs = model(x_dwt, x_stft)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            
            preds = (torch.sigmoid(outputs) > threshold).int()
            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
            
    all_preds = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)
    
    val_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    
    return val_loss/len(loader), val_f1

In [ ]:
if __name__ == '__main__':
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Используем: {device}")

    num_classes = 72 
    model = ECGMultiModalNet(num_classes=num_classes).to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(), lr=0.001)
    
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.7, patience=4, threshold=0.001, threshold_mode='abs')
    
    full_dataset = ECGDataset(data_dir='D:/processed_data')
    subset_indices = list(range(2000)) 
    # small_dataset = torch.utils.data.Subset(full_dataset, subset_indices)
    # вместо него потом просто full_dataset
    
    train_size = int(0.8 * len(full_dataset))
    val_size = len(full_dataset) - train_size
    train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size])

    train_loader = DataLoader(
        train_dataset, 
        batch_size=50, 
        shuffle=False, # для качества True, для скорости False (связано с чтением файлов и подачей их на обучение)
        num_workers=0,
        pin_memory=True if torch.cuda.is_available() else False
    )
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)
    
    # 6. Запуск обучения
    train_model(train_loader, val_loader, device, epochs=15)